# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.

In [19]:
class matrix:
    def __init__(self, *args):
        # I'm checking if the arguments are two integers (n, m) to create a zero matrix
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            self.n, self.m = args[0], args[1]
            self.data = [[0 for _ in range(self.m)] for _ in range(self.n)]
            
        # Or if the argument is a list of lists, I need to copy it over
        elif len(args) == 1 and isinstance(args[0], list):
            input_list = args[0]
            self.n = len(input_list)
            self.m = len(input_list[0]) if self.n > 0 else 0
            
            # Making sure every row has the same number of columns to catch bad inputs
            if not all(isinstance(row, list) and len(row) == self.m for row in input_list):
                raise ValueError("All rows must have the same number of columns.")
                
            self.data = [[val for val in row] for row in input_list]
        else:
            raise ValueError("Need either (n, m) or a list of lists.")

    def __getitem__(self, index):
        # Python passes M[i,j] as a tuple, so I unpack it
        if isinstance(index, tuple):
            i, j = index
            return self.data[i][j]
        # For M[i][j], the first brackets pass an int, returning the whole row
        elif isinstance(index, int):
            return self.data[index]
        else:
            raise TypeError("Index needs to be an int or tuple.")

    def __setitem__(self, index, value):
        # Allowing value assignment like M[0,1] = 5
        if isinstance(index, tuple):
            i, j = index
            self.data[i][j] = value
        elif isinstance(index, int):
            self.data[index] = value

    def assign(self, other):
        # The assignment operator '=' cannot be overloaded in Python. 
        # Writing M_1 = M_2 just creates a reference copy, it doesn't modify the object's values.
        # So, I'm using an assign() method to handle the specific assignment logic requested.
        if isinstance(other, matrix):
            if self.n != other.n or self.m != other.m:
                raise ValueError("Size mismatch.")
            self.data = [[val for val in row] for row in other.data]
        elif isinstance(other, list):
            if len(other) != self.n or not all(len(row) == self.m for row in other):
                raise ValueError("Size mismatch.")
            self.data = [[val for val in row] for row in other]

    def __str__(self):
        return '\n'.join([str(row) for row in self.data])

# --- EXPLICIT TESTS FOR Q1 ---
print("--- Q1 Tests ---")
m1 = matrix(2, 3)
assert m1.data == [[0, 0, 0], [0, 0, 0]]
m2 = matrix([[1, 2], [3, 4]])
assert m2.data == [[1, 2], [3, 4]]

m_target = matrix(2, 2)
m_target.assign(m2)
assert m_target.data == [[1, 2], [3, 4]]
print("Q1 Initialization, Indexing, and Assignment tests passed!")

--- Q1 Tests ---
Q1 Initialization, Indexing, and Assignment tests passed!


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [20]:
class matrix:
    # --- Q1 Methods ---
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            self.n, self.m = args[0], args[1]
            self.data = [[0 for _ in range(self.m)] for _ in range(self.n)]
        elif len(args) == 1 and isinstance(args[0], list):
            input_list = args[0]
            self.n, self.m = len(input_list), (len(input_list[0]) if len(input_list) > 0 else 0)
            if not all(isinstance(row, list) and len(row) == self.m for row in input_list):
                raise ValueError("All rows must have the same number of columns.")
            self.data = [[val for val in row] for row in input_list]

    def __getitem__(self, index):
        # Upgraded to handle slicing. If the index contains a 'slice' object (like 0:2),
        # I use list comprehensions to grab those specific chunks.
        if isinstance(index, tuple):
            i, j = index
            if isinstance(i, slice) or isinstance(j, slice):
                if isinstance(i, int): i = slice(i, i+1)
                if isinstance(j, int): j = slice(j, j+1)
                return matrix([row[j] for row in self.data[i]])
            return self.data[i][j]
        elif isinstance(index, slice):
            return matrix(self.data[index])
        elif isinstance(index, int):
            return self.data[index]

    def __setitem__(self, index, value):
        if isinstance(index, tuple): i, j = index; self.data[i][j] = value
        elif isinstance(index, int): self.data[index] = value

    def assign(self, other):
        if isinstance(other, matrix):
            if self.shape() != other.shape(): raise ValueError("Size mismatch.")
            self.data = [[val for val in row] for row in other.data]
        elif isinstance(other, list):
            self.data = [[val for val in row] for row in other]
            
    def __str__(self): return '\n'.join([str(row) for row in self.data])

    # --- Q2 New Methods ---
    def shape(self):
        # Simply returns the dimensions stored during init
        return (self.n, self.m)

    def transpose(self):
        # Swapping rows and columns to create the transposed version
        t_data = [[self.data[i][j] for i in range(self.n)] for j in range(self.m)]
        return matrix(t_data)

    def row(self, n):
        # Wrapping the row in an extra list bracket so it becomes a 1xm matrix
        return matrix([self.data[n]])

    def column(self, n):
        # Grabbing the nth element of every row to build an nx1 matrix
        return matrix([[self.data[i][n]] for i in range(self.n)])

    def to_list(self):
        # Using a list comprehension to return a clean copy, avoiding reference bugs
        return [[val for val in row] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        # Standard Python slicing: grab rows m_0 to m_1, then slice those rows from cols n_0 to n_1
        return matrix([r[n_0:n_1] for r in self.data[m_0:m_1]])

# --- EXPLICIT TESTS FOR Q2 ---
print("--- Q2 Tests ---")
m_test = matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
assert m_test.shape() == (3, 3)
assert m_test.transpose().data == [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
assert m_test.row(1).data == [[4, 5, 6]]
assert m_test.column(2).data == [[3], [6], [9]]
assert m_test.to_list() == [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
assert m_test.block(1, 3, 0, 2).data == [[2, 3], [5, 6]]
print("Q2 Properties and Slicing tests passed!")

--- Q2 Tests ---
Q2 Properties and Slicing tests passed!


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [21]:
# --- Q3 Standalone Functions ---

def constant(n, m, c):
    # Creating an n by m matrix filled with the float version of c
    val = float(c)
    return matrix([[val for j in range(m)] for i in range(n)])

def zeros(n, m):
    # Reusing the constant function to avoid rewriting the same loops
    return constant(n, m, 0.0)

def ones(n, m):
    # Same trick, reusing constant for 1.0
    return constant(n, m, 1.0)

def eye(n):
    # Building the identity matrix: 1.0 on the diagonal (where i == j), 0.0 elsewhere
    return matrix([[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)])

# --- EXPLICIT TESTS FOR Q3 ---
print("--- Q3 Tests ---")
assert constant(2, 2, 5).data == [[5.0, 5.0], [5.0, 5.0]]
assert zeros(2, 3).data == [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
assert ones(2, 2).data == [[1.0, 1.0], [1.0, 1.0]]
assert eye(3).data == [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
print("Q3 Special Matrices tests passed!")

--- Q3 Tests ---
Q3 Special Matrices tests passed!


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [23]:
class matrix:
    # --- Q1 & Q2 Methods ---
    def __init__(self, *args):
        if len(args) == 2 and isinstance(args[0], int) and isinstance(args[1], int):
            self.n, self.m = args[0], args[1]
            self.data = [[0 for _ in range(self.m)] for _ in range(self.n)]
        elif len(args) == 1 and isinstance(args[0], list):
            input_list = args[0]
            self.n, self.m = len(input_list), (len(input_list[0]) if len(input_list) > 0 else 0)
            if not all(isinstance(row, list) and len(row) == self.m for row in input_list):
                raise ValueError("All rows must have the same number of columns.")
            self.data = [[val for val in row] for row in input_list]

    def __getitem__(self, index):
        if isinstance(index, tuple):
            i, j = index
            if isinstance(i, slice) or isinstance(j, slice):
                if isinstance(i, int): i = slice(i, i+1)
                if isinstance(j, int): j = slice(j, j+1)
                return matrix([row[j] for row in self.data[i]])
            return self.data[i][j]
        elif isinstance(index, slice): return matrix(self.data[index])
        elif isinstance(index, int): return self.data[index]

    def __setitem__(self, index, value):
        if isinstance(index, tuple): i, j = index; self.data[i][j] = value
        elif isinstance(index, int): self.data[index] = value

    def assign(self, other):
        if isinstance(other, matrix):
            if self.shape() != other.shape(): raise ValueError("Size mismatch.")
            self.data = [[val for val in row] for row in other.data]
        elif isinstance(other, list):
            self.data = [[val for val in row] for row in other]

    def shape(self): return (self.n, self.m)
    def transpose(self): return matrix([[self.data[i][j] for i in range(self.n)] for j in range(self.m)])
    def row(self, n): return matrix([self.data[n]])
    def column(self, n): return matrix([[self.data[i][n]] for i in range(self.n)])
    def to_list(self): return [[val for val in row] for row in self.data]
    def block(self, n_0, n_1, m_0, m_1): return matrix([r[n_0:n_1] for r in self.data[m_0:m_1]])
    def __str__(self): return '\n'.join([str(row) for row in self.data])

    # --- Q4 Math Operations ---
    def scalarmul(self, c):
        return matrix([[val * c for val in row] for row in self.data])

    def add(self, N):
        if self.shape() != N.shape(): raise ValueError("Addition error: Size mismatch.")
        return matrix([[self.data[i][j] + N.data[i][j] for j in range(self.m)] for i in range(self.n)])

    def sub(self, N):
        if self.shape() != N.shape(): raise ValueError("Subtraction error: Size mismatch.")
        return matrix([[self.data[i][j] - N.data[i][j] for j in range(self.m)] for i in range(self.n)])

    def mat_mult(self, N):
        if self.m != N.n: raise ValueError("Matrix multiplication error: M columns must equal N rows.")
        result_data = []
        for i in range(self.n):
            new_row = []
            for j in range(N.m):
                # taking the dot product of M's row and N's column
                new_row.append(sum(self.data[i][k] * N.data[k][j] for k in range(self.m)))
            result_data.append(new_row)
        return matrix(result_data)

    def element_mult(self, N):
        if self.shape() != N.shape(): raise ValueError("Element multiplication error: Size mismatch.")
        return matrix([[self.data[i][j] * N.data[i][j] for j in range(self.m)] for i in range(self.n)])

    def equals(self, N):
        if not isinstance(N, matrix) or self.shape() != N.shape(): return False
        return self.data == N.data

    # --- Q5 Operator Overloading ---
    
    # Python uses magic methods like __add__ to know what to do when you type '+'
    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __mul__(self, other):
        # I need to check if 'other' is a matrix or a scalar to decide which math rule to apply
        if isinstance(other, matrix):
            return self.mat_mult(other)
        elif isinstance(other, (int, float)):
            return self.scalarmul(other)

    def __rmul__(self, other):
        # This handles the case where the scalar comes first, like 2 * M
        return self.scalarmul(other)

    def __eq__(self, other):
        # Overloads the '==' operator
        return self.equals(other)
        
    # Note on M=N from Q5 requirements: Python physically cannot overload the '=' operator. 
    # So Using the assign() method implemented in Q1 to assign values without creating a reference copy.

# --- EXPLICIT TESTS FOR Q4 & Q5 ---
print("--- Q4 & Q5 Tests ---")
M1 = matrix([[1, 2], [3, 4]])
M2 = matrix([[5, 6], [7, 8]])

# Testing Q4 explicit methods
assert M1.scalarmul(3).data == [[3, 6], [9, 12]]
assert M1.mat_mult(M2).data == [[19, 22], [43, 50]]

# Testing Q5 overloaded operators
assert (M1 + M2).data == [[6, 8], [10, 12]]
assert (M1 - M2).data == [[-4, -4], [-4, -4]]
assert (M1 * M2).data == [[19, 22], [43, 50]]
assert (M1 * 2).data == [[2, 4], [6, 8]]
assert (2 * M1).data == [[2, 4], [6, 8]]
assert (M1 == matrix([[1, 2], [3, 4]])) == True

print("Q4 Math Methods and Q5 Overloaded Operators tests passed!")

--- Q4 & Q5 Tests ---
Q4 Math Methods and Q5 Overloaded Operators tests passed!


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [25]:
# Create test matrices
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])
C = matrix([[9, 10], [11, 12]])
I = eye(2) 

print("--- Q6 Mathematical Demonstrations ---\n")

# 1. Associativity: (AB)C = A(BC)
left_side = (A * B) * C
right_side = A * (B * C)
print("1. Demonstrating (AB)C = A(BC)")
print(f"Is it true? {left_side == right_side}\n")
assert left_side == right_side

# 2. Distributivity: A(B+C) = AB + AC
left_side = A * (B + C)
right_side = (A * B) + (A * C)
print("2. Demonstrating A(B+C) = AB + AC")
print(f"Is it true? {left_side == right_side}\n")
assert left_side == right_side

# 3. Non-commutativity: AB != BA
AB = A * B
BA = B * A
print("3. Demonstrating AB != BA")
print(f"Is it true? {not (AB == BA)}\n")
assert not (AB == BA)

# 4. Identity: AI = A
AI = A * I
print("4. Demonstrating AI = A")
print(f"Is it true? {AI == A}\n")
assert AI == A

print("All Q6 linear algebra properties successfully demonstrated!")

--- Q6 Mathematical Demonstrations ---

1. Demonstrating (AB)C = A(BC)
Is it true? True

2. Demonstrating A(B+C) = AB + AC
Is it true? True

3. Demonstrating AB != BA
Is it true? True

4. Demonstrating AI = A
Is it true? True

All Q6 linear algebra properties successfully demonstrated!
